# Stacked BiLSTM Sweep


## Setup



In [1]:
import os
import sys
import base64
import subprocess
from pathlib import Path

ENVIRONMENT = "colab" if "google.colab" in sys.modules else "kaggle"
root = Path("/content" if ENVIRONMENT == "colab" else "/kaggle/working")
repo_dir = Path(os.environ.get("UA_NER_REPO_DIR", root / "UA-Locations-NER-Deep-Learning"))

if not repo_dir.exists():
    if ENVIRONMENT == "colab":
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    else:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ.copy()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraheader",
               GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}")
    try:
        subprocess.run(["git", "clone", "-b", "stacked",
                        "https://github.com/dkalchenko01/UA-Locations-NER-Deep-Learning.git",
                        str(repo_dir)], env=env, check=True)
    finally:
        del token, auth, env

sys.path.insert(0, str(repo_dir))
print(repo_dir)



/content/UA-Locations-NER-Deep-Learning


In [2]:
%pip install -q -r "$repo_dir/requirements.txt"



In [3]:
import gc
import json
import platform
import uuid
from importlib.metadata import version

import pandas as pd
import torch
from tokenizers import Tokenizer

from src.data_io import (get_data_dir, load_manifest, load_data_splits,
                         save_json, save_predictions, file_sha256)
from src.dataset import prepare_ner_records, prepare_inference_records, make_loader, LABEL_TO_ID
from src.model import BiLSTMTagger
from src.metrics import competition_scores
from src.preprocessing import clean_locations_v3, serialize_spans
from src.training import set_seed, train_epoch, predict, save_checkpoint

import wandb
from time import perf_counter
from src.experiment_tracking import start_run, log_epoch, log_evaluation, finish_run


## W&B login


In [4]:
os.environ["WANDB_BASE_URL"] = "https://api.forge.coreweave.com"
os.environ["WANDB_DISABLE_CODE"] = "true"
os.environ["WANDB_CONSOLE"] = "off"

if ENVIRONMENT == "colab":
    from google.colab import userdata
    wandb.login(key=userdata.get("WANDB_API_KEY"), relogin=True)
else:
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"), relogin=True)



/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.forge.coreweave.com.
wandb: No netrc file found, creating one.
wandb: Appending key for api.forge.coreweave.com to your netrc file: /root/.netrc
wandb: Currently logged in as: daldoshyna (daldoshyna-kyiv-school-of-economics) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


## Configuration



In [5]:
config = {
    "model": "bilstm", "architecture": "stacked", "embedding_dim": 128, "hidden_size": 256,
    "num_layers": 3, "dropout": 0.2, "seed": 17, "batch_size": 128,
    "learning_rate": 1e-3, "epochs": 5, "gradient_clip": 1.0,
    "max_padded_tokens": 16384, "prepare_batch_size": 2000,
    "sources": ["uk", "ru"], "optimizer": "Adam",
    "tokenizer": "bpe_bytelevel.json", "selection_metric": "uk_comp_f1",
    "label_to_id": LABEL_TO_ID, "max_sequence_length": None,
    "early_stopping": False, "environment": ENVIRONMENT,
    "validation_version": "v1", "metric_version": "kaggle_original_regex_v1",
    "postprocessing_version": "raw", "upload_checkpoint": False,
}
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
set_seed(config["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
run_id = "sweep_setup_" + pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
run_dir = root / "outputs" / run_id
run_dir.mkdir(parents=True, exist_ok=False)
splits_dir = repo_dir / "data/splits"
tokenizer_path = repo_dir / "artifacts/tokenizers" / config["tokenizer"]

provenance = {
    "run_id": run_id, "seed": config["seed"], "python": platform.python_version(),
    "git_revision": subprocess.run(["git", "-C", str(repo_dir), "rev-parse", "HEAD"],
                                   capture_output=True, text=True).stdout.strip() or None,
    "git_status": subprocess.run(["git", "-C", str(repo_dir), "status", "--short"],
                                 capture_output=True, text=True).stdout.strip(),
    "packages": {name: version(name) for name in ["torch", "tokenizers", "numpy", "pandas", "pyarrow", "tqdm", "wandb"]},
    "device": str(device), "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
    "tokenizer_sha256": file_sha256(tokenizer_path),
    "manifest_sha256": file_sha256(splits_dir / "validation_manifest.parquet"),
    "source_sha256": {path.name: file_sha256(path) for path in sorted((repo_dir / "src").glob("*.py"))},
    "notebook_sha256": file_sha256(repo_dir / "notebooks/04_bilstm_sweep.ipynb"),
}
save_json(run_dir / "config.json", config)
save_json(run_dir / "provenance.json", provenance)
print(device, run_dir)



cuda /content/outputs/sweep_setup_20261111T220037Z_ba230c7f


## Data



In [6]:
tokenizer = Tokenizer.from_file(str(tokenizer_path))
tokenizer.no_padding()
tokenizer.no_truncation()
pad_id = tokenizer.token_to_id("[PAD]")
data_dir = get_data_dir(ENVIRONMENT)
manifest = load_manifest(splits_dir)
display(pd.crosstab(manifest["source"], manifest["split"]))
del manifest

train_records, val_records, alignment = [], [], []
for source in config["sources"]:
    source_splits = load_data_splits(ENVIRONMENT, splits_dir, sources=(source,))
    for split in ("train", "val"):
        frame = source_splits.pop(f"{source}_{split}")[["text", "loc_markers"]]
        records, skipped = prepare_ner_records(tokenizer, frame, source, split,
                                               batch_size=config["prepare_batch_size"])
        (train_records if split == "train" else val_records).extend(records)
        alignment.append({"source": source, "split": split, "total": len(frame),
                          "retained": len(records), "alignment_failures": len(skipped)})
        save_json(run_dir / f"{source}_{split}_skipped.json", skipped)
        del frame, records, skipped
        gc.collect()
    del source_splits
alignment = pd.DataFrame(alignment)
display(alignment)
alignment.to_csv(run_dir / "alignment.csv", index=False)



100%|██████████| 811M/811M [00:46<00:00, 18.4MB/s]

Extracting files...


split,train,val
source,,
ru,100000,49369
uk,805983,201498


Prepare uk/train:   0%|          | 0/403 [00:00<?, ?it/s]

Prepare uk/val:   0%|          | 0/101 [00:00<?, ?it/s]

Prepare ru/train:   0%|          | 0/50 [00:00<?, ?it/s]

Prepare ru/val:   0%|          | 0/25 [00:00<?, ?it/s]

,source,split,total,retained,alignment_failures
0,uk,train,805983,804381,1602
1,uk,val,201498,201498,361
2,ru,train,100000,99890,110
3,ru,val,49369,49369,44


In [7]:
loader_options = {
    "pad_id": pad_id, "batch_size": config["batch_size"], "seed": config["seed"],
    "max_padded_tokens": config["max_padded_tokens"], "pin_memory": device.type == "cuda"
}
train_loader = make_loader(train_records, shuffle=True, **loader_options)
val_loader = make_loader(val_records, **loader_options)



## W&B Sweep


In [8]:
from src.sweeps import run_sweep_trial
from src.experiment_tracking import DEFAULT_ENTITY, DEFAULT_PROJECT

sweep_base = {**config, "num_layers": 3, "hidden_size": 256, "epochs": 5,
              "selection_metric": "uk_comp_f1"}
sweep_config = {
    "name": "bilstm-3layer-lr-dropout",
    "method": "grid",
    "metric": {"name": "best/uk_comp_f1", "goal": "maximize"},
    "parameters": {
        "num_layers": {"value": 3},
        "hidden_size": {"value": 256},
        "epochs": {"value": 5},
        "dropout": {"values": [0.2, 0.4]},
        "learning_rate": {"values": [3e-4, 1e-3, 3e-3]},
    },
}
def train_sweep():
    run_sweep_trial(
        base_config=sweep_base, provenance=provenance,
        tokenizer=tokenizer, train_loader=train_loader, val_loader=val_loader,
        val_records=val_records, device=device, root=root, repo_dir=repo_dir,
    )


In [9]:
sweep_id = wandb.sweep(sweep_config, entity=DEFAULT_ENTITY, project=DEFAULT_PROJECT)
save_json(root / "outputs" / f"sweep_{sweep_id}.json", {
    "sweep_id": sweep_id, "config": sweep_config,
})
print(sweep_id)


Create sweep with ID: kd5bc9ih
Sweep URL: https://app.forge.coreweave.com/daldoshyna-kyiv-school-of-economics/ua-location-ner/sweeps/kd5bc9ih
kd5bc9ih


In [10]:
wandb.finish()
wandb.agent(sweep_id, function=train_sweep, count=6,
            entity=DEFAULT_ENTITY, project=DEFAULT_PROJECT)


wandb: Agent Starting Run: uyc2doke with config:
wandb: 	dropout: 0.2
wandb: 	epochs: 5
wandb: 	hidden_size: 256
wandb: 	learning_rate: 0.0003
wandb: 	num_layers: 3
wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /root/.netrc.


Train:   0%|          | 0/7110 [00:01<?, ?it/s]

Predict:   0%|          | 0/1973 [00:00<?, ?it/s]

sweep_uyc2doke {'epoch': 1, 'train_loss': 0.046709154988142286, 'epoch_seconds': 329.9935206470001, 'uk_val_loss': 0.024639739546270058, 'ru_val_loss': 0.034528000660478786, 'combined_val_loss': 0.02711778845618065, 'uk_comp_f1': 0.83566329, 'ru_comp_f1': 0.81189434, 'combined_comp_f1': 0.82981995, 'clean_uk_comp_f1': 0.83545496, 'clean_ru_comp_f1': 0.81193154, 'clean_combined_comp_f1': 0.82966804}


Train:   0%|          | 0/7119 [00:00<?, ?it/s]

Predict:   0%|          | 0/1973 [00:00<?, ?it/s]

... (92 outputs left out)

Train:   0%|          | 0/7114 [00:00<?, ?it/s]

Predict:   0%|          | 0/1973 [00:00<?, ?it/s]

sweep_wfpoe1mo {'epoch': 5, 'train_loss': 0.01483828852072305, 'epoch_seconds': 332.8552560629996, 'uk_val_loss': 0.018213264715448454, 'ru_val_loss': 0.02271432421731525, 'combined_val_loss': 0.019341253293340722, 'uk_comp_f1': 0.87555278, 'ru_comp_f1': 0.88741011, 'combined_comp_f1': 0.87855362, 'clean_uk_comp_f1': 0.87534549, 'clean_ru_comp_f1': 0.88741011, 'clean_combined_comp_f1': 0.87839886}


epoch,▁▃▅▆█
time/epoch_seconds,▅█▂▄▁
train/loss,█▂▁▁▁
val_v1/clean_v3/combined_comp_f1,▁█▅▇▄
val_v1/clean_v3/ru_comp_f1,▁▇█▇▆
val_v1/clean_v3/uk_comp_f1,▁█▂▆▂
val_v1/combined_comp_f1,▁█▆▇▄
val_v1/combined_loss,█▁▂▇▆
val_v1/ru_comp_f1,▁▇█▇▆
val_v1/ru_loss,█▁▁▅▄
+2,...
